#### Import Libraries


In [ ]:
# Basic libraries
import os
import glob
import random
import time

# Data handling
import numpy as np
import pandas as pd

# Image processing
from PIL import Image

# Visualization
import matplotlib.pyplot as plt

# ML utilities
from sklearn.model_selection import train_test_split

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Progress bar
from tqdm import tqdm

# For reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [2]:
from google.colab import drive
drive.mount('/content/drive')                                                                                           

Mounted at /content/drive


#### Data import 


In [3]:
# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# Additional GPU info
if device.type == "cuda":
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())
    
    
# Set dataset path (UPDATE THIS PATH)
dataset_path = "/content/drive/MyDrive/extracted_data/train"  # e.g., "data/train"

# Get all image file paths
image_paths = glob.glob(os.path.join(dataset_path, "*.jpg"))

print("Total images found:", len(image_paths))
print("Sample paths:", image_paths[:5])    

Using device: cuda
GPU Name: Tesla T4
GPU Count: 1
Total images found: 25000
Sample paths: ['/content/drive/MyDrive/extracted_data/train/cat.9107.jpg', '/content/drive/MyDrive/extracted_data/train/cat.9086.jpg', '/content/drive/MyDrive/extracted_data/train/cat.9119.jpg', '/content/drive/MyDrive/extracted_data/train/cat.9075.jpg', '/content/drive/MyDrive/extracted_data/train/cat.9109.jpg']


#### Label Encoding


In [4]:
data = []

for path in image_paths:
    filename = os.path.basename(path)
    label_text = filename.split(".")[0]
    
    label = 0 if label_text == "cat" else 1
    
    data.append([path, label_text, label])

# Create DataFrame
df = pd.DataFrame(data, columns=["filepath", "label_text", "label"])

print(df.head())
print("\nClass distribution:\n", df["label_text"].value_counts())

                                            filepath label_text  label
0  /content/drive/MyDrive/extracted_data/train/ca...        cat      0
1  /content/drive/MyDrive/extracted_data/train/ca...        cat      0
2  /content/drive/MyDrive/extracted_data/train/ca...        cat      0
3  /content/drive/MyDrive/extracted_data/train/ca...        cat      0
4  /content/drive/MyDrive/extracted_data/train/ca...        cat      0

Class distribution:
 label_text
cat    12500
dog    12500
Name: count, dtype: int64


#### Data Sepration 

In [5]:
# Separate cat and dog images
cat_df = df[df["label_text"] == "cat"]
dog_df = df[df["label_text"] == "dog"]

print("Total Cats:", len(cat_df))
print("Total Dogs:", len(dog_df))

Total Cats: 12500
Total Dogs: 12500


#### Sampling equal no. of images 


In [6]:
# Define sample size per class
sample_size = 6000  # you can change to 4000 or 5000 if needed

cat_sample = cat_df.sample(n=sample_size, random_state=42)
dog_sample = dog_df.sample(n=sample_size, random_state=42)

# Combine
df_sampled = pd.concat([cat_sample, dog_sample]).reset_index(drop=True)

# Shuffle
df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)

print("Sampled dataset size:", len(df_sampled))
print(df_sampled["label_text"].value_counts())

Sampled dataset size: 12000
label_text
cat    6000
dog    6000
Name: count, dtype: int64


#### Train Validate Test

In [7]:
# First split: Train vs Temp (val + test)
train_df, temp_df = train_test_split(
    df_sampled,
    test_size=0.3,
    stratify=df_sampled["label"],
    random_state=42
)

# Second split: Validation vs Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

Train size: 8400
Validation size: 1800
Test size: 1800


#### Preprocessing Pipeline

In [ ]:
IMG_SIZE = 128  # you can change to 224 for ResNet

def preprocess_image(path):
    # Read, resize, and convert to RGB
    img = Image.open(path).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)

    # Normalize
    img = np.asarray(img, dtype=np.float32) / 255.0

    return img

#### Change shape: (H, W, C) → (C, H, W)

In [9]:
def image_to_tensor(img):
    # Convert to numpy array (already is)
    
    # Change shape: (H, W, C) → (C, H, W)
    img = np.transpose(img, (2, 0, 1))
    
    # Convert to torch tensor
    tensor = torch.tensor(img, dtype=torch.float32)
    
    return tensor

#### Image to Tensor for Entire Dataset


In [10]:
def process_dataframe(df):
    images = []
    labels = []
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img = preprocess_image(row["filepath"])
        tensor = image_to_tensor(img)
        
        images.append(tensor)
        labels.append(row["label"])
    
    X = torch.stack(images)
    y = torch.tensor(labels)
    
    return X, y

# Process datasets
X_train, y_train = process_dataframe(train_df)
X_val, y_val = process_dataframe(val_df)
X_test, y_test = process_dataframe(test_df)

print("Train tensor shape:", X_train.shape)
print("Validation tensor shape:", X_val.shape)
print("Test tensor shape:", X_test.shape)

 79%|███████▉  | 6673/8400 [38:14<09:53,  2.91it/s]  


KeyboardInterrupt: 

#### Saving Preprocessed Images into .pt files

In [ ]:
torch.save(X_train, "X_train.pt")
torch.save(y_train, "y_train.pt")

torch.save(X_val, "X_val.pt")
torch.save(y_val, "y_val.pt")

torch.save(X_test, "X_test.pt")
torch.save(y_test, "y_test.pt")

print("Data saved successfully!")

Data saved successfully!


#### Data Loading

In [ ]:
X_train = torch.load("X_train.pt")
y_train = torch.load("y_train.pt")

X_val = torch.load("X_val.pt")
y_val = torch.load("y_val.pt")

X_test = torch.load("X_test.pt")
y_test = torch.load("y_test.pt")

print("Data loaded successfully!")

FileNotFoundError: [Errno 2] No such file or directory: 'X_train.pt'

#### Data Loader

In [ ]:
# Create TensorDataset
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

print("Train dataset size:", len(train_dataset))

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Number of train batches:", len(train_loader))

Train dataset size: 8400
Number of train batches: 132


#### Testing for One batch

In [ ]:
# Get one batch
images, labels = next(iter(train_loader))

print("Batch shape:", images.shape)
print("Labels shape:", labels.shape)

Batch shape: torch.Size([64, 3, 128, 128])
Labels shape: torch.Size([64])


#### Moving image to gpu


In [ ]:
images = images.to(device)
labels = labels.to(device)

print("Device of images:", images.device)

Device of images: cuda:0


#### Filter apply


In [ ]:
edge_filter = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
])
sharpen_filter = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
])
blur_filter = np.ones((3,3)) / 9



In [ ]:
model = SimpleCNN().to(device)

print(model)

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (fc1): Linear(in_features=32768, out_features=128, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)


#### Max pooling


In [ ]:
# Simple max pooling function
def max_pooling(img, size=2):
    h, w = img.shape
    
    # Ensure dimensions divisible by size
    h_new = h // size
    w_new = w // size
    
    pooled = np.zeros((h_new, w_new))
    
    for i in range(h_new):
        for j in range(w_new):
            pooled[i, j] = np.max(
                img[i*size:(i+1)*size, j*size:(j+1)*size]
            )
    
    return pooled

pooled_img = max_pooling(edge_img, size=2)

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.imshow(edge_img, cmap='gray')
plt.title("Before Pooling")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(pooled_img, cmap='gray')
plt.title("After Pooling")
plt.axis("off")

plt.show()

print("Original shape:", edge_img.shape)
print("Pooled shape:", pooled_img.shape)

NameError: name 'edge_img' is not defined

#### Pytorch

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Convolution Blocks
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        
        # Fully Connected Layers
        self.fc1 = nn.Linear(128 * 16 * 16, 128)  # depends on input size (128x128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 2)
    
    def forward(self, x):
        # Block 1
        x = self.pool(self.relu(self.conv1(x)))
        
        # Block 2
        x = self.pool(self.relu(self.conv2(x)))
        
        # Block 3
        x = self.pool(self.relu(self.conv3(x)))
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully connected
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

#### Total Parameters

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 4287938
Trainable parameters: 4287938


#### Loss Function & Optimizers


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

#### Training 

In [ ]:
num_epochs = 30

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    model.train()
    
    running_loss = 0
    running_acc = 0
    
    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        running_acc += calculate_accuracy(outputs, labels)
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = running_acc / len(train_loader)
    
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    
    # Validation
    model.eval()
    val_loss = 0
    val_acc = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            val_acc += calculate_accuracy(outputs, labels)
    
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)
    
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

#### Losss Visualization 

In [ ]:
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
model.eval()

test_loss = 0
test_acc = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        
        test_acc += (preds == labels).sum().item() / len(labels)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss /= len(test_loader)
test_acc /= len(test_loader)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Cat", "Dog"],
            yticklabels=["Cat", "Dog"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds, target_names=["Cat", "Dog"]))

In [ ]:
def show_predictions(images, labels, preds, n=6):
    plt.figure(figsize=(12,6))
    
    for i in range(n):
        img = images[i].permute(1,2,0).cpu()
        
        plt.subplot(2,3,i+1)
        plt.imshow(img)
        plt.title(f"True: {labels[i].item()} | Pred: {preds[i].item()}")
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

# Get one batch
images, labels = next(iter(test_loader))
images = images.to(device)

outputs = model(images)
_, preds = torch.max(outputs, 1)

show_predictions(images, labels, preds)

In [ ]:
def show_wrong_predictions(loader, model, n=6):
    model.eval()
    
    wrong_images = []
    wrong_labels = []
    wrong_preds = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            for i in range(len(labels)):
                if preds[i] != labels[i]:
                    wrong_images.append(images[i].cpu())
                    wrong_labels.append(labels[i].cpu())
                    wrong_preds.append(preds[i].cpu())
                    
                    if len(wrong_images) == n:
                        break
            if len(wrong_images) == n:
                break
    
    show_predictions(
        torch.stack(wrong_images),
        torch.stack(wrong_labels),
        torch.stack(wrong_preds),
        n
    )

show_wrong_predictions(test_loader, model)

## Single-Image Inference and Streamlit Testing

The trained CNN expects an RGB image resized to `128 x 128` pixels, scaled to the `[0, 1]` range, and rearranged from `(H, W, C)` to `(C, H, W)`. The output classes are `0 = Cat` and `1 = Dog`.

The next cell saves the model weights together with the class names and image size. The resulting `simple_cnn_cats_dogs.pt` file is used by the Streamlit app.


In [ ]:
checkpoint_path = "simple_cnn_cats_dogs.pt"
class_names = ["Cat", "Dog"]

model.eval()
torch.save(
    {
        "model_state_dict": model.cpu().state_dict(),
        "class_names": class_names,
        "image_size": IMG_SIZE,
    },
    checkpoint_path,
)
model.to(device)

print(f"Checkpoint saved to: {checkpoint_path}")

In [ ]:
model.eval()
sample_images, _ = next(iter(test_loader))

with torch.inference_mode():
    sample_outputs = model(sample_images.to(device))
    sample_probabilities = torch.softmax(sample_outputs, dim=1)
    sample_predictions = sample_probabilities.argmax(dim=1)

print("Predictions:", [class_names[index] for index in sample_predictions[:6].cpu().tolist()])
print("First prediction probabilities:", sample_probabilities[0].cpu().tolist())
print("Probability sum:", sample_probabilities[0].sum().item())

## Run the Streamlit Image Tester

After running the checkpoint export cell, open a terminal in this project folder and run:

```bash
streamlit run app.py
```

Upload a `.jpg`, `.jpeg`, `.png`, or `.webp` image. The app applies the same preprocessing used during training and displays the predicted class with both class probabilities.